# 08 - RAG Document Ingestion with Gemini and Pinecone

## Learning objective

Build the **ingestion half** of a Retrieval-Augmented Generation pipeline:

```text
mediumblog1.txt
-> load the document
-> split it into chunks
-> create 1,536-dimensional Gemini embeddings
-> store the chunks and vectors in Pinecone
```

This notebook uses:

- `TextLoader` for the plain-text source file
- `CharacterTextSplitter` for chunking
- `GoogleGenerativeAIEmbeddings` instead of OpenAI embeddings
- `PineconeVectorStore` for vector storage
- Deterministic chunk IDs so rerunning the notebook updates the same records instead of creating duplicates

## Required Pinecone index configuration

```text
Index name: langchain-course-rag
Vector type: Dense
Dimension: 1536
Metric: cosine
Capacity mode: Serverless
Cloud: AWS
Region: us-east-1
```

> Run the notebook cells in order. Cells that call Gemini or Pinecone are clearly marked.


## 1. Optional package installation

Run this cell only if an import fails. After installation, restart the notebook kernel.

```text
Kernel -> Restart Kernel
```


In [ ]:
# %pip install -U \
#     langchain-community \
#     langchain-google-genai \
#     langchain-pinecone \
#     langchain-text-splitters \
#     pinecone \
#     python-dotenv \
#     truststore


## 2. Check the active Python kernel

The executable should point to your `nlp_env` Conda environment.


In [1]:
import sys


print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 3. Configure corporate SSL before importing API clients

`truststore` makes Python use the Windows certificate store. It should be injected before creating HTTPS clients.


In [2]:
import truststore
truststore.inject_into_ssl()
print("Windows trust store injected into Python SSL.")

Windows trust store injected into Python SSL.


## 4. Import the ingestion components

The source is a plain `.txt` file, so `TextLoader` is simpler than `UnstructuredLoader` and avoids unnecessary parsing dependencies.


In [3]:
import hashlib
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import CharacterTextSplitter
from pinecone import Pinecone

print("Imports completed successfully.")

C:\Users\inkk1r04\AppData\Local\Temp\ipykernel_24928\2415509912.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Imports completed successfully.


## 5. Locate the document and `.env` file

`mediumblog1.txt` should be in the same folder as this notebook.

The code searches for `.env` in two places:

1. The notebook's current folder
2. The sibling `langchain-course` folder used by the earlier course setup


In [13]:
CURRENT_DIR = Path.cwd().resolve()
DOCUMENT_PATH = CURRENT_DIR / "mediumblog1.txt"
DOCUMENT_ORIGINAL_SOURCE = "https://medium.com/@EjiroOnose/vector-database-what-is-it-and-why-you-should-know-it-ae7e7dca82a4"


env_candidates = [
    CURRENT_DIR / ".env",
    CURRENT_DIR.parent / "langchain-course" / ".env",
]


ENV_PATH = next(
    (
        candidate
        for candidate in env_candidates
        if candidate.exists()
    ),
    None,
)


print("Current directory:", CURRENT_DIR)
print("Document path:", DOCUMENT_PATH)
print("Document exists:", DOCUMENT_PATH.exists())
print("Environment file found:", ENV_PATH is not None)


if ENV_PATH is not None:
    print("Environment file path:", ENV_PATH)


Current directory: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter
Document path: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\mediumblog1.txt
Document exists: True
Environment file found: True
Environment file path: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course\.env


## 6. Load environment variables safely

The notebook checks only whether secrets are present. It never prints secret values.

The expected `.env` entries are:

```env
GOOGLE_API_KEY=your-google-api-key
PINECONE_API_KEY=your-pinecone-api-key
INDEX_NAME=langchain-course-rag
```


In [5]:
if ENV_PATH is None:
    raise FileNotFoundError(
        "Could not find .env in the notebook folder "
        "or the sibling langchain-course folder."
    )


load_dotenv(dotenv_path=ENV_PATH)


required_variables = [
    "GOOGLE_API_KEY",
    "PINECONE_API_KEY",
    "INDEX_NAME",
]


for variable_name in required_variables:
    print(
        f"{variable_name} loaded:",
        bool(os.getenv(variable_name)),
    )


missing_variables = [
    variable_name
    for variable_name in required_variables
    if not os.getenv(variable_name)
]


if missing_variables:
    raise ValueError(
        "Missing environment variables: "
        + ", ".join(missing_variables)
    )


if not DOCUMENT_PATH.exists():
    raise FileNotFoundError(
        f"Document not found: {DOCUMENT_PATH}"
    )


GOOGLE_API_KEY loaded: True
PINECONE_API_KEY loaded: True
INDEX_NAME loaded: True


## 7. Define the shared configuration

The Gemini output dimensionality must exactly match the Pinecone index dimension.


In [6]:
INDEX_NAME = os.environ["INDEX_NAME"]
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 1536
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 0
NAMESPACE = "mediumblog1"


print("Index name:", INDEX_NAME)
print("Embedding model:", EMBEDDING_MODEL)
print("Embedding dimension:", EMBEDDING_DIMENSION)
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)
print("Namespace:", NAMESPACE)


Index name: langchain-course-rag
Embedding model: gemini-embedding-001
Embedding dimension: 1536
Chunk size: 1000
Chunk overlap: 0
Namespace: mediumblog1


## 8. Connect to Pinecone and inspect the existing index

> **This cell makes Pinecone API requests but does not create embeddings.**

It confirms that the expected index already exists and that its dimension and similarity metric match the notebook configuration.


In [7]:
pinecone_client = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)


existing_index_names = [
    index_info["name"]
    for index_info in pinecone_client.list_indexes()
]


print("Existing Pinecone indexes:", existing_index_names)


if INDEX_NAME not in existing_index_names:
    raise ValueError(
        f"Pinecone index '{INDEX_NAME}' does not exist."
    )


index_description = pinecone_client.describe_index(
    INDEX_NAME
)


print("Index dimension:", index_description.dimension)
print("Index metric:", index_description.metric)
print("Index host:", index_description.host)
print("Index ready:", index_description.status["ready"])


if index_description.dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Pinecone dimension is {index_description.dimension}, "
        f"but Gemini is configured for {EMBEDDING_DIMENSION}."
    )


if index_description.metric != "cosine":
    raise ValueError(
        f"Expected cosine metric, found {index_description.metric}."
    )


Existing Pinecone indexes: ['langchain-course-rag']
Index dimension: 1536
Index metric: cosine
Index host: langchain-course-rag-upeditj.svc.aped-4627-b74a.pinecone.io
Index ready: True


## 9. Load the text document

`TextLoader` converts the file into LangChain `Document` objects containing:

- `page_content`: the document text
- `metadata`: information such as the source path

This cell is local and makes no external API request.


In [8]:
loader = TextLoader(
    file_path=str(DOCUMENT_PATH),
    encoding="utf-8",
)


documents = loader.load()


print("Loaded Document objects:", len(documents))
print("Source metadata:", documents[0].metadata)
print("Character count:", len(documents[0].page_content))
print("Text preview:")
print(documents[0].page_content[:1000])


Loaded Document objects: 1
Source metadata: {'source': 'C:\\Users\\inkk1r04\\OneDrive - Kellogg Company\\Desktop\\Work\\0. DS Capability\\Courses\\Langchain\\langchain-course-jupyter\\mediumblog1.txt'}
Character count: 16197
Text preview:

Search
Write

Emarco
Vector Database: What is it and why you should know it?
Ejiro Onose
Ejiro Onose

·
Follow

9 min read
·
Dec 22, 2023
100







“If 2021 was the year of graph databases, 2023 is the year of vector databases” — Chip Huen.

Why? you ask...

Well, it’s because Generative AI and Large Language Models (LLMs) are now popular and one of the best ways to handle LLM data is with a vector database because vector databases provide the ideal infrastructure for managing the complex, high-dimensional data that LLMs produce and thrive on.

In this article, I’ll discuss what vector databases are, how they work, and some excellent vector database tools you should check out.

Before we dive into vector databases, let’s first understand what a vect

## 10. Split the document into chunks

The instructor uses:

```text
chunk_size = 1000 characters
chunk_overlap = 0 characters
```

We retain those values to stay aligned with the course.

This cell is local and makes no external API request.


In [9]:
text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)


chunks = text_splitter.split_documents(
    documents
)


print(f"Created {len(chunks)} chunks")


chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]


print("Shortest chunk:", min(chunk_lengths))
print("Longest chunk:", max(chunk_lengths))
print(
    "Average chunk length:",
    round(sum(chunk_lengths) / len(chunk_lengths), 1),
)


Created a chunk of size 1180, which is longer than the specified 1000
Created a chunk of size 1058, which is longer than the specified 1000


Created 20 chunks
Shortest chunk: 52
Longest chunk: 1180
Average chunk length: 806.0


## 11. Inspect chunk content and metadata

This is an important quality check before spending embedding quota.


In [11]:
for chunk_number, chunk in enumerate(
    chunks[:3],
    start=1,
):
    print("=" * 80)
    print("Chunk number:", chunk_number)
    print("Length:", len(chunk.page_content))
    print("Metadata:", chunk.metadata)
    print("Content:")
    print(chunk.page_content[:1200])
    print()


Chunk number: 1
Length: 954
Metadata: {'source': 'C:\\Users\\inkk1r04\\OneDrive - Kellogg Company\\Desktop\\Work\\0. DS Capability\\Courses\\Langchain\\langchain-course-jupyter\\mediumblog1.txt'}
Content:
Search
Write

Emarco
Vector Database: What is it and why you should know it?
Ejiro Onose
Ejiro Onose

·
Follow

9 min read
·
Dec 22, 2023
100

“If 2021 was the year of graph databases, 2023 is the year of vector databases” — Chip Huen.

Why? you ask...

Well, it’s because Generative AI and Large Language Models (LLMs) are now popular and one of the best ways to handle LLM data is with a vector database because vector databases provide the ideal infrastructure for managing the complex, high-dimensional data that LLMs produce and thrive on.

In this article, I’ll discuss what vector databases are, how they work, and some excellent vector database tools you should check out.

Before we dive into vector databases, let’s first understand what a vector is.

What is a Vector?
In machine lear

## 12. Add useful metadata and deterministic IDs

Each chunk receives metadata useful for filtering and inspection.

A deterministic SHA-256-based ID is generated from the source name, chunk number, and chunk text. If the same notebook is rerun with unchanged chunks, Pinecone updates the same records instead of adding random duplicates.


In [18]:
chunk_ids = []

for chunk_number, chunk in enumerate(chunks):

    # Replace all loader-generated metadata with
    # explicitly approved metadata.
    chunk.metadata = {
        "source": DOCUMENT_ORIGINAL_SOURCE,
        "source_file": DOCUMENT_PATH.name,
        "chunk_number": chunk_number,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimension": EMBEDDING_DIMENSION,
    }

    id_source = (
        f"{DOCUMENT_PATH.name}|"
        f"{chunk_number}|"
        f"{chunk.page_content}"
    )

    chunk_id = hashlib.sha256(
        id_source.encode("utf-8")
    ).hexdigest()

    chunk_ids.append(chunk_id)


print("Number of chunk IDs:", len(chunk_ids))
print("First chunk ID:", chunk_ids[0])
print("First chunk metadata:", chunks[0].metadata)

Number of chunk IDs: 20
First chunk ID: 33ea5a091decaa616bb76255434ca997dec476752054f2cbd55a4722a9e5fb32
First chunk metadata: {'source': 'https://medium.com/@EjiroOnose/vector-database-what-is-it-and-why-you-should-know-it-ae7e7dca82a4', 'source_file': 'mediumblog1.txt', 'chunk_number': 0, 'embedding_model': 'gemini-embedding-001', 'embedding_dimension': 1536}


## 13. Initialize Gemini embeddings

Creating the object does not call the Gemini API.

The explicit `output_dimensionality=1536` setting ensures compatibility with the existing Pinecone index.


In [19]:
embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    output_dimensionality=EMBEDDING_DIMENSION,
)


print("Embedding object:", type(embeddings).__name__)
print("Model:", EMBEDDING_MODEL)
print("Requested dimension:", EMBEDDING_DIMENSION)


Embedding object: GoogleGenerativeAIEmbeddings
Model: gemini-embedding-001
Requested dimension: 1536


## 14. Test one Gemini embedding before ingestion

> **This cell makes one Gemini embedding request.**

It confirms that the API key works and that Gemini returns exactly 1,536 values.


In [20]:
test_vector = embeddings.embed_query(
    "Test the Gemini embedding dimension for the RAG course."
)


print("Embedding Python type:", type(test_vector))
print("Embedding dimension:", len(test_vector))
print("First five values:", test_vector[:5])


if len(test_vector) != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Expected {EMBEDDING_DIMENSION} values, "
        f"received {len(test_vector)}."
    )


Embedding Python type: <class 'list'>
Embedding dimension: 1536
First five values: [-0.008085896, 0.0001239274, -0.011749642, -0.08400953, 0.00846961]


## 15. Create the Pinecone vector-store connection

This connects the existing Pinecone index to the Gemini embedding object.

The namespace keeps this tutorial dataset logically separated from other datasets that may later share the same index.


In [21]:
pinecone_index = pinecone_client.Index(
    host=index_description.host
)


vector_store = PineconeVectorStore(
    index=pinecone_index,
    embedding=embeddings,
    namespace=NAMESPACE,
)


print("Vector store created.")
print("Namespace:", NAMESPACE)


Vector store created.
Namespace: mediumblog1


## 16. Upload chunks to Pinecone

> **This cell makes Gemini embedding requests and Pinecone write requests. Avoiding Reruning to avoid api calls**

It embeds every chunk and upserts the resulting records using deterministic IDs.

Run this cell only after all earlier validation cells succeed.


In [22]:
# print(
#     f"Embedding and uploading {len(chunks)} chunks "
#     f"to index '{INDEX_NAME}'..."
# )


# uploaded_ids = vector_store.add_documents(
#     documents=chunks,
#     ids=chunk_ids,
# )


# print("Upload completed.")
# print("Uploaded record count:", len(uploaded_ids))
# print("First uploaded ID:", uploaded_ids[0])


Embedding and uploading 20 chunks to index 'langchain-course-rag'...
Upload completed.
Uploaded record count: 20
First uploaded ID: 33ea5a091decaa616bb76255434ca997dec476752054f2cbd55a4722a9e5fb32


## 17. Inspect Pinecone index statistics

> **This cell makes one Pinecone API request.**

Pinecone may be eventually consistent, so recently uploaded records can take a short time to appear in statistics or searches.


In [23]:
index_statistics = pinecone_index.describe_index_stats()


print("Total vector count:", index_statistics.total_vector_count)
print("Dimension:", index_statistics.dimension)
print("Namespaces:", index_statistics.namespaces)


Total vector count: 20
Dimension: 1536
Namespaces: {'mediumblog1': {'vector_count': 20}}


## 18. Run a retrieval sanity check

> **This cell makes one Gemini query-embedding request and one Pinecone search request.**

The same Gemini model and 1,536-dimensional configuration are used for both document ingestion and query retrieval.


In [24]:
query = "What is the main topic discussed in the article?"


results = vector_store.similarity_search_with_score(
    query=query,
    k=3,
)


print("Query:", query)


for result_number, (document, score) in enumerate(
    results,
    start=1,
):
    print("=" * 80)
    print("Result:", result_number)
    print("Similarity score:", score)
    print("Metadata:", document.metadata)
    print("Content:")
    print(document.page_content[:1200])
    print()


Query: What is the main topic discussed in the article?
Result: 1
Similarity score: 0.644983768
Metadata: {'chunk_number': 18.0, 'embedding_dimension': 1536.0, 'embedding_model': 'gemini-embedding-001', 'source': 'https://medium.com/@EjiroOnose/vector-database-what-is-it-and-why-you-should-know-it-ae7e7dca82a4', 'source_file': 'mediumblog1.txt'}
Content:
ChatGPT prompts
47 stories
·
1439 saves
Sparse embedding or BM25?
InfiniFlow
InfiniFlow

Sparse embedding or BM25?
Since the open-sourcing of Infinity, it has received a wide positive response from the community. Regarding the essential RAG technology we…
7 min read
·
Feb 13, 2024
16

Experimenting with Vector Databases: Chromadb, Pinecone, Weaviate and Pgvector
Vishnu Sivan
Vishnu Sivan

in

CoinsBench

Experimenting with Vector Databases: Chromadb, Pinecone, Weaviate and Pgvector
Vector databases are specialized systems designed for storing, managing, and searching embedding vectors. The widespread adoption of…
10 min read
·
Nov 14, 

## 19. Optional cleanup cell

The ingestion code uses deterministic IDs, so you normally do not need to delete records before rerunning it.

If you intentionally want to remove every record in this tutorial namespace, uncomment and run the cell below.

> **Warning:** This permanently deletes all records in the `mediumblog1` namespace.


In [ ]:
# pinecone_index.delete(
#     delete_all=True,
#     namespace=NAMESPACE,
# )
#
# print(
#     f"Deleted all records from namespace '{NAMESPACE}'."
# )


## 20. What this notebook accomplished

```text
mediumblog1.txt
        ↓
TextLoader
        ↓
LangChain Document
        ↓
CharacterTextSplitter
        ↓
Document chunks
        ↓
Gemini gemini-embedding-001
1,536 values per chunk
        ↓
Pinecone index: langchain-course-rag
namespace: mediumblog1
```

## Important rules for the retrieval notebook

The retrieval half must use:

```text
Embedding model: gemini-embedding-001
Output dimensionality: 1536
Index name: langchain-course-rag
Namespace: mediumblog1
```

Changing the embedding model or dimension requires re-embedding and re-indexing the documents.
